# Playing with the Moran engine

`simulate(W, n_mutants, fitness, n_repeats, max_steps, seed)` runs a multi-type Birth-death Moran process on the graph `W`.
`W[i, j]` is the weight of the edge i -> j (`PopulationGraph.csr()` builds it). The wild type has fitness 1.

Each run returns `winner` (mutant type k, `-1` wild type, `-2` censored), `steps` and `censored`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from moran_graphs import PopulationGraph as pg, simulate
from moran_graphs.engine import WILD_TYPE, CENSORED

## One graph, one call

In [ ]:
g = pg.mammalian_lung_graph(depth=4)
g.draw()

res = simulate(g.csr(), n_mutants=[1], fitness=[1.5], n_repeats=10_000, max_steps=10**8, seed=0)
res

Fixation probability, and mean fixation time **conditional on fixation** (censored runs are excluded from both).

In [ ]:
def summarize(res, k=0):
    won = res["winner"] == k
    done = ~res["censored"]
    return {
        "rho": won.sum() / done.sum(),
        "mean_time_given_fixation": res["steps"][won].mean() if won.any() else np.nan,
        "censored": int(res["censored"].sum()),
    }

summarize(res)

## Fixation probability vs r, compared across topologies
All graphs have N = 31. The dashed line is the well-mixed Moran formula.

In [ ]:
N = 31
graphs = {
    "complete": pg.complete_graph(N),
    "cycle": pg.cycle_graph(N),
    "star": pg.star_graph(N),
    "mammalian lung": pg.mammalian_lung_graph(depth=4),
    "random (E=40)": pg.random_graph(N, 40, seed=1),
}
rs = np.linspace(0.8, 2.0, 7)
n_repeats = 5_000

table = {
    name: [summarize(simulate(g.csr(), [1], [r], n_repeats, 10**8, seed=i)) for i, r in enumerate(rs)]
    for name, g in graphs.items()
}

In [ ]:
fig, (ax_rho, ax_time) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, rows in table.items():
    ax_rho.plot(rs, [row["rho"] for row in rows], "o-", label=name)
    ax_time.plot(rs, [row["mean_time_given_fixation"] for row in rows], "o-", label=name)

with np.errstate(invalid="ignore"):  # 0/0 at r = 1, replaced by its limit 1/N
    moran = (1 - 1 / rs) / (1 - rs ** -N)
moran[np.isclose(rs, 1)] = 1 / N
ax_rho.plot(rs, moran, "k--", label="Moran formula")
ax_rho.set(xlabel="r", ylabel="fixation probability")
ax_time.set(xlabel="r", ylabel="mean steps | fixation", yscale="log")
ax_rho.legend()
plt.tight_layout()

## Several mutant types at once
A has a 10% advantage, B cannot reproduce, C has a 100% advantage. Each starts on 2 random nodes.

In [ ]:
res = simulate(pg.complete_graph(50).csr(), n_mutants=[1, 1, 1], fitness=[1.1, 0.0, 2.0],
               n_repeats=20_000, max_steps=10**8, seed=7)
labels = {WILD_TYPE: "wild type", 0: "A (1.1)", 1: "B (0.0)", 2: "C (2.0)", CENSORED: "censored"}
winners, counts = np.unique(res["winner"], return_counts=True)
{labels[w]: c / len(res["winner"]) for w, c in zip(winners, counts)}

## Directed graphs
On the directed star (hub -> leaves) only the hub can spread, so a mutant fixes iff it starts on the hub: rho = 1/N for every r.
A reproducer with no out-neighbors wastes the step.

In [ ]:
for r in [0.5, 1.0, 3.0]:
    print(r, summarize(simulate(pg.star_graph(10, directed=True).csr(), [1], [r], 20_000, 10**8, seed=3)))

## Faster Moran: compute time
`skip_null_steps=True` only simulates steps where the reproducer and its target differ in type, and draws how many null steps were skipped in between, so the output has the same distribution (checked below).
Below, both engines run the same 300 runs on graphs of about 127 nodes. Each time is a single measurement, so expect some noise, mostly in the fastest cases.

In [ ]:
import time

palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
bench_graphs = {
    "complete": pg.complete_graph(127),
    "cycle": pg.cycle_graph(127),
    "star": pg.star_graph(127),
    "mammalian lung": pg.mammalian_lung_graph(depth=6),
    "avian": pg.avian_graph(8, 15),
    "fish": pg.fish_graph(7, 6),
    "random (E=190)": pg.random_graph(127, 190, seed=1),
}
bench_rs = [1.05, 1.25, 1.5, 2.0, 3.0]
n_runs = 300


def timed(W, r, skip):
    t = time.perf_counter()
    res = simulate(W, [1], [r], n_runs, 10**10, seed=0, skip_null_steps=skip)
    return time.perf_counter() - t, res


bench = []
for name, g in bench_graphs.items():
    W = g.csr()
    for r in bench_rs:
        t_plain, res = timed(W, r, False)
        t_skip, _ = timed(W, r, True)
        bench.append({
            "graph": name, "N": W.shape[0], "r": r,
            "plain_ms": 1e3 * t_plain / n_runs, "skip_ms": 1e3 * t_skip / n_runs,
            "speedup": t_plain / t_skip,
            "null_share": 1 - res["n_changes"].sum() / res["steps"].sum(),
        })

print(f"{'graph':16s} {'N':>4s} {'r':>5s} {'plain ms/run':>13s} {'skip ms/run':>12s} {'speed-up':>9s} {'null steps':>11s}")
for b in bench:
    print(f"{b['graph']:16s} {b['N']:4d} {b['r']:5.2f} {b['plain_ms']:13.3f} {b['skip_ms']:12.3f}"
          f" {b['speedup']:8.1f}x {b['null_share']:10.1%}")

In [ ]:
fig, (ax_r, ax_null) = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for color, name in zip(palette, bench_graphs):
    rows = [b for b in bench if b["graph"] == name]
    speedup = [b["speedup"] for b in rows]
    ax_r.plot([b["r"] for b in rows], speedup, "o-", color=color, lw=2, ms=6, label=name)
    ax_null.plot([b["null_share"] for b in rows], speedup, "o", color=color, ms=8,
                 markeredgecolor="white", markeredgewidth=1.5, label=name)

for ax in (ax_r, ax_null):
    ax.axhline(1, color="#888888", lw=1, ls="--")
    ax.text(ax.get_xlim()[0], 1, " break-even", va="bottom", color="#555555", fontsize=9)
    ax.set_yscale("log")
    ax.grid(color="#dddddd", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
ax_r.set(xlabel="r", ylabel="speed-up  (plain time / skipping time)",
         title="Speed-up of skipping null steps, N ≈ 127")
ax_null.set(xlabel="share of null steps in the plain engine",
            title="It pays off when most steps change nothing")
ax_null.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
fig.legend(*ax_r.get_legend_handles_labels(), loc="center right", frameon=False, fontsize=9)
plt.tight_layout(rect=(0, 0, 0.87, 1))

### Same distribution? Check it yourself
Both engines on the same lung, compared by their empirical CDFs. `steps` is Moran time; `n_changes` counts steps in which a node changed type, so `steps - n_changes` are the null steps the skipping engine never simulates.
The p-values are two-sample Kolmogorov-Smirnov tests (scipy). The Rust test `cargo test engines_agree -- --nocapture` (run in `rust/engine/`) does the same on three small graphs and prints a table.

In [ ]:
from scipy.stats import ks_2samp

W, r, reps = pg.mammalian_lung_graph(depth=5).csr(), 1.2, 20_000
plain = simulate(W, [1], [r], reps, 10**10, seed=1)
skip = simulate(W, [1], [r], reps, 10**10, seed=2, skip_null_steps=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, (ending, who) in enumerate([("fixation", 0), ("extinction", WILD_TYPE)]):
    for ax, col in zip(axes[row], ["steps", "n_changes"]):
        for label, res in [("plain", plain), ("skip null steps", skip)]:
            x = np.sort(res[col][res["winner"] == who])
            ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", label=f"{label} (n={len(x)})")
        a, b = (res[col][res["winner"] == who] for res in (plain, skip))
        ax.set(xscale="log", xlabel=col, ylabel="CDF", title=f"{col}, runs ending in {ending}: KS p = {ks_2samp(a, b).pvalue:.2f}")
        ax.legend()
plt.tight_layout()
print("rho plain", (plain["winner"] == 0).mean(), " rho skip", (skip["winner"] == 0).mean())
print("share of null steps (plain):", 1 - plain["n_changes"].sum() / plain["steps"].sum())

## Speed
Steps per second is the number to watch; runs per second depends on how long fixation takes.

In [ ]:
import time

W = pg.mammalian_lung_graph(depth=9).csr()  # N = 1023
t = time.perf_counter()
res = simulate(W, [1], [1.5], 500, 10**9, seed=11)
dt = time.perf_counter() - t
print(f"N={W.shape[0]}: {500 / dt:.0f} runs/s, {res['steps'].sum() / dt / 1e6:.0f}M steps/s")